In [ ]:
import ase.io
import ase
import numpy as np
import schnetpack as spk
from ase.units import Bohr
import os
from ase.db import connect
import schnarc
import torch
from tqdm import tqdm

In [ ]:

def readfile(filename):
    try:
        f = open(filename)
        out = f.readlines()
        f.close()
    except IOError:
        print('File %s does not exist!' % (filename))
        sys.exit(12)
    return out

def writefile(filename, content):
    # content can be either a string or a list of strings
    try:
        f = open(filename, 'w')
        if isinstance(content, list):
            for line in content:
                f.write(line)
        elif isinstance(content, str):
            f.write(content)
        else:
            print('Content %s cannot be written to file!' % (content))
            sys.exit(13)
        f.close()
    except IOError:
        print('Could not write to file %s!' % (filename))
        sys.exit(14)
        
        
def eformat(f, prec, exp_digits):
    '''Formats a float f into scientific notation with prec number of decimals and exp_digits number of exponent digits.

    String looks like:
    [ -][0-9]\\.[0-9]*E[+-][0-9]*

    Arguments:
    1 float: Number to format
    2 integer: Number of decimals
    3 integer: Number of exponent digits

    Returns:
    1 string: formatted number'''

    s = "% .*e" % (prec, f)
    mantissa, exp = s.split('e')
    return "%sE%+0*d" % (mantissa, exp_digits + 1, int(exp))


def itmult(states):

    for i in range(len(states)):
        if states[i] < 1:
            continue
        yield i + 1
    return


def itnmstates(states):

    for i in range(len(states)):
        if states[i] < 1:
            continue
        for k in range(i + 1):
            for j in range(states[i]):
                yield i + 1, j + 1, k - i / 2.
    return

In [ ]:

Ntrajs = 300
device = torch.device('cuda:0')
pred_data = spk.data.AtomsData("../Initconds/DBO.db")

pred_loader = spk.data.AtomsLoader(pred_data,
                                   batch_size=Ntrajs,
                                   num_workers=0,
                                   pin_memory=True)

ii = 0
for batch in tqdm(pred_loader,ncols=120):
    
    if ii<1:
        
        bb={k:v.to(device) for k,v in batch.items()}
    
    ii += 1

  0%|                                                                                            | 0/11 [00:00<?, ?it/s]/home/zhulx/.conda/envs/pysharc/lib/python3.9/site-packages/schnetpack/data/atoms.py:710: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /opt/conda/conda-bld/pytorch_1670525539683/work/torch/csrc/utils/tensor_numpy.cpp:199.)
  torch_properties[pname] = torch.LongTensor(prop)
100%|███████████████████████████████████████████████████████████████████████████████████| 11/11 [00:52<00:00,  4.79s/it]


In [ ]:

keys_list = ['has_gradients', 'gradients', '_atomic_numbers', '_positions', '_neighbors', '_cell', '_cell_offset', '_idx', '_neighbor_mask', '_atom_mask']

    
for kk in keys_list:
    if kk in ['has_gradients', 'gradients', '_positions']: #['energy', 'forces', 'dipoles', 'has_forces']:
        del bb[kk]



torch.save(bb, f"../Initconds/initdb_dict_{Ntrajs}.pth")